# NenuFAR Solar Imaging Workflow (v0.20 notebook)

This notebook demonstrates a **notebook-driven UI workflow (ipywidgets)** for NenuFAR solar interferometric imaging:

- **Step0 (optional):** scan / select candidate SBs
- **Step1–4:** MS preparation → ROI cut → calibration (DP3) → imaging (WSClean) → FITS
- **Step4 quicklook:** generate PNG quicklooks / optional movies from `*-image.fits`
- **Step5A/5B:** quiet-Sun based **ionospheric offset (WCS shift)** solve + apply (write corrected FITS)
- **Step5C:** centroid extraction (2D Gaussian fit within an ROI) + tables + optional movie

---

## Where to run / access
This workflow is intended to run on the **Nançay computing environment** (e.g., `nancep`), with access to NenuFAR data paths and the required compute permissions.
Access may require approval from the **NenuFAR KP11 (Solar) team**.

---

## Two external files (not in this repo)
Some Step1–4 actions require two external resources (paths below are Nançay-specific):

- Container image: `linc_latest.sif`
- Calibration DB: `CasA.sourcedb`

If you need access to them, please contact the author / KP11 Solar team.

---

## Output folders (important)
Each step writes outputs under a `step*_outputs_YYYYMMDD/` folder (event-date tag).
Step5 writes:
- `step_iocorrect_outputs_YYYYMMDD/` (Step5A/5B)
- `step_iocentroid_outputs_YYYYMMDD/` (Step5C)

Check the printed notebook logs for the exact output paths for your run.

In [1]:
import importlib, nenufar_ui
importlib.reload(nenufar_ui)
from nenufar_ui import load_and_select_sb
from nenufar_ui import run_step1_ui
from nenufar_ui import run_step2_zoom_ui
from nenufar_ui import run_step3_calib_ui
from nenufar_ui import run_step4_wsclean_ui
from nenufar_ui import run_step4_quicklook_ui
from nenufar_ui import run_step5a_iocorrect_solve_ui
from nenufar_ui import run_step5b_iocorrect_apply_ui
from nenufar_ui import run_step5c_centroid_ui

# Scan & SB Selection

## Purpose
Locate SUN and CAS_A events for a given date, align SBs, and generate a processing plan (JSON).

## What it does
- Find SUN_TRACKING event
- Find all CAS_A_TRACKING candidates (pre & post)
- Match SBs by frequency
- Allow user SB + calibrator selection
- Save selected SB list to JSON

## Output
`selected_sb_pair_list.json`

Contains:
- selected_sb
- sun_ms
- casa_pre_ms
- casa_post_ms
- casa_ms (if chosen)

⚠ No DP3 processing is done here.
This step only builds the processing plan.

**How to use**
1. Set:
   - `BASE` (NenuFAR data root, e.g. `/databf/nenufar-nri/LT11`)
   - `WORKROOT` (your working directory for outputs)
2. Run the cell to open the widget UI.
3. Adjust Year/Month/Day, frequency range, search filters, then click **Run**.
4. The output table helps you identify SB(s) to process in Step1–4.

**Output**
- A selected SB plan JSON is typically written under a `...SUN_TRACKING/selected_sb_pair_list.json` folder (see the printed output).

In [2]:
import sys
sys.path.insert(0, "/data/jzhang/20240310TYPEII_work")

from nenufar_ui import load_and_select_sb

BASE = "/databf/nenufar-nri/LT11"
WORKROOT = "/data/jzhang/nenufar_workflows"

load_and_select_sb(BASE, WORKROOT)

Output()

# Step-1 — Prepare Calibrator & Sun MS

## Goal
Prepare clean working copies of:
- CasA (calibrator)
- Sun (target)

No gain solving is done here.

---

## 1A — CasA: AOFlagger + Average

DP3:
steps=[flag,averager]
flag.type=aoflagger
averager.timestep=1
averager.freqstep=1

Output:
CasA_SBxxx_prep.MS

Purpose:
- Remove RFI
- Standardize resolution

---

## 1B — CasA: Preflag bad baselines (in-place)

DP3:
steps=[flag]
flag.type=preflagger
flag.baseline='MR102NEN&&*;MR103NEN&&*'

Purpose:
- Remove unstable baselines
- Stabilize calibration

---

## 1C — Sun: Clear flags → new copy

DP3:
steps=[flag]
flag.type=preflagger
flag.mode=clear
flag.baseline='*&&*'

Output:
SUN_SBxxx_prep.MS

Purpose:
- Reset flags
- Protect raw data
- Prepare for calibration

---

## Result per SB
CasA_SBxxx_prep.MS
SUN_SBxxx_prep.MS
(log files)

Step-1 = data hygiene stage.
Safe to re-run.

In [3]:
import sys
sys.path.insert(0, "/data/jzhang/nenufarIMG_workflow")

from nenufar_ui import run_step1_ui

plan = "/data/jzhang/nenufar_workflows/20240310_101000_20240310_135000_SUN_TRACKING/selected_sb_pair_list.json"
run_step1_ui(plan, out_root="/data/jzhang/nenufar_workflows/step1_outputs_20240310")

## Step-2 — Zooming in (ROI Time Selection)

**Goal**  
Extract a time window (Region of Interest, ROI) from the prepared SUN MS produced in Step-1.

**Input**
- `SUN_{SB}_prep.MS` (from Step-1)
- Start time (e.g. `2024/03/10/12:00:00`)
- End time (e.g. `2024/03/10/12:20:00`)

**Operation (DP3)**
```bash
DP3 msin=SUN_{SB}_prep.MS \
     msin.starttime='YYYY/MM/DD/HH:MM:SS' \
     msin.endtime='YYYY/MM/DD/HH:MM:SS' \
     steps=[] \
     msout=SUN_{SB}_ROI.MS


**Output:**

SUN_{SB}_ROI.MS

**Located in:**

step2_outputs_YYYYMMDD/ROI/SBxxx/

In [4]:
import sys
sys.path.insert(0, "/data/jzhang/nenufarIMG_workflow")

from nenufar_ui import run_step2_zoom_ui

plan = "/data/jzhang/nenufar_workflows/20240310_101000_20240310_135000_SUN_TRACKING/selected_sb_pair_list.json"

run_step2_zoom_ui(
    plan_json_path=plan,
    step1_root="/data/jzhang/nenufar_workflows/step1_outputs_20240310",
    out_root="/data/jzhang/nenufar_workflows/step2_outputs_20240310",
    default_start="2024/03/10/12:08:00",
    default_end="2024/03/10/12:38:00",
)

## Step-3 — DI Calibration (CasA → Sun ROI)

### Goal
Use the **CasA calibrator** to derive per-antenna gain solutions, then **apply** them to the **Sun ROI MS**, producing a calibrated data column (**CORR_NO_BEAM**).

> Note: **applybeam is intentionally NOT used** for NenuFAR solar imaging (cannot apply beam in this workflow).  
> We stop at: **gaincal (predict/solve) + applycal**.

---

### Inputs
For each selected subband `SBxxx.MS`:

1. **Calibrator MS (CasA)**  
   From **Step-1 outputs** (selected by `dropdown/pre/post`):
   - `step1_root/SBxxx/CasA_SBxxx_prep.MS`

2. **Target MS (Sun ROI)**  
   From **Step-2 outputs**:
   - `step2_root/ROI/SBxxx/SUN_SBxxx_ROI.MS`

3. **Calibrator sky model database**
   - `CasA.sourcedb` (pre-generated, provided by you)

---

### What it runs (per SB)
#### 3A) `gaincal` on CasA (solve gains)
- Generates a parset:
  - `step3_root/SBxxx/SBxxx_gaincal.parset`
- Runs DP3 gaincal, producing a solution table:
  - `step3_root/SBxxx/CasA_SBxxx_prep.MS/instrument`

#### 3B) `applycal` on Sun ROI (apply gains)
- Generates a parset:
  - `step3_root/SBxxx/SBxxx_applycal.parset`
- Runs DP3 applycal on:
  - `step2_root/ROI/SBxxx/SUN_SBxxx_ROI.MS`
- Writes calibrated visibilities into a **new data column**:
  - **`CORR_NO_BEAM`**  
  (in-place inside the same Sun ROI MS)

---

### Outputs
Per `SBxxx` you will get:

**Files (in Step-3 folder)**
- `step3_root/SBxxx/SBxxx_gaincal.parset`
- `step3_root/SBxxx/SBxxx_applycal.parset`
- logs:
  - `01_gaincal.log`
  - `02_applycal.log`

**Data product (in Step-2 ROI MS)**
- `step2_root/ROI/SBxxx/SUN_SBxxx_ROI.MS`
  - now contains a new column: **`CORR_NO_BEAM`**

---

### Why this matters
After Step-3, the Sun ROI dataset is calibrated (direction-independent, no beam) and ready for imaging steps that expect:
- stable gain-corrected visibilities
- `CORR_NO_BEAM` as the calibrated column

---

### Quick checks
**1) confirm CORR_NO_BEAM exists**
```bash
taql "select COLNAME() from /path/to/SUN_SBxxx_ROI.MS"

In [6]:
import sys
sys.path.insert(0, "/data/jzhang/nenufarIMG_workflow")

from nenufar_ui import run_step3_calib_ui

plan = "/data/jzhang/nenufar_workflows/20240310_101000_20240310_135000_SUN_TRACKING/selected_sb_pair_list.json"

run_step3_calib_ui(
    plan_json_path=plan,
    step1_root="/data/jzhang/nenufar_workflows/step1_outputs_20240310",
    step2_root="/data/jzhang/nenufar_workflows/step2_outputs_20240310",
    out_root="/data/jzhang/nenufar_workflows/step3_outputs_20240310",
    sourcedb="/data/jzhang/CasA.sourcedb",
)

## Step4 — Imaging (WSClean) → FITS images

This step runs **WSClean** imaging on the ROI MS produced in Step2 (and calibrated products from Step3 if configured in the workflow).

**How to use**
1. Confirm your inputs:
   - `plan_json_path` points to your `selected_sb_pair_list.json`
   - `step2_root` points to `step2_outputs_YYYYMMDD`
   - `out_root` points to `step4_outputs_YYYYMMDD`
2. Run the cell to open the Step4 widget UI.
3. In the UI, select SB(s) and set WSClean imaging parameters:
   - image size / scale, weighting, robust, niter, auto-mask, etc.
4. Click **Run Step-4 (WSClean)**.

**Outputs**
- FITS images are written under:
  - `step4_outputs_YYYYMMDD/SBxxx/*-image.fits`
- WSClean logs are printed in the notebook and saved under the corresponding SB output folder.

In [7]:
import sys
sys.path.insert(0, "/data/jzhang/nenufarIMG_workflow")

from nenufar_ui import run_step4_wsclean_ui

plan = "/data/jzhang/nenufar_workflows/20240310_101000_20240310_135000_SUN_TRACKING/selected_sb_pair_list.json"

run_step4_wsclean_ui(
    plan_json_path=plan,
    step2_root="/data/jzhang/nenufar_workflows/step2_outputs_20240310",
    out_root="/data/jzhang/nenufar_workflows/step4_outputs_20240310",
)

## Step4 Quicklook — PNG quicklooks / optional movies from FITS

This tool reads Step4 FITS images (`*-image.fits`) and generates:
- quicklook PNG(s)
- optional movie (MP4 if `ffmpeg` exists; otherwise GIF fallback)

**How to use**
1. Set `step4_root` to your `step4_outputs_YYYYMMDD` folder.
2. Run the cell to open the widget UI.
3. In the UI:
   - choose the SB
   - multi-select one or more `*-image.fits`
   - set crop half-width (arcsec), CLim percentiles, contours
   - optionally enable **Make video** and set FPS
4. Click **Generate Quicklooks**.

**Outputs**
- PNGs:
  - `step4_outputs_YYYYMMDD/quicklook/SBxxx/*.png`
- Optional movie:
  - `step4_outputs_YYYYMMDD/quicklook/SBxxx/*quicklook.mp4` (or `.gif`)
- Log:
  - `step4_outputs_YYYYMMDD/quicklook/SBxxx/quicklook.log`

**Note**
Only `*-image.fits` are considered science images here (other FITS products are ignored).

In [8]:
import sys
sys.path.insert(0, "/data/jzhang/nenufarIMG_workflow")

from nenufar_ui import run_step4_quicklook_ui

run_step4_quicklook_ui(
    step4_root="/data/jzhang/nenufar_workflows/step4_outputs_20240310",
    out_root="/data/jzhang/nenufar_workflows/step4_outputs_20240310/quicklook",
    default_sb="SB359",
)

## Step5A — Solve ionospheric offset (WCS shift) from a Quiet Sun frame

Step5A estimates an **ionospheric refraction offset** using a selected **Quiet Sun** image.
The idea: the Quiet Sun centroid (within an ROI) should be near the solar-disc center in HPC.
The measured centroid offset is converted into a WCS shift solution for later correction.

**How to use**
1. Set `step4_root` to your Step4 outputs.
2. Run the cell to open the Step5A widget UI.
3. In the UI:
   - choose the SB
   - select a **Quiet Sun** FITS frame (before burst onset)
   - define an ROI (arcsec) covering the Quiet Sun emission
   - click **Solve Step5A**
4. The UI will display a diagnostic plot and print the derived offset.

**Outputs**
- Solution JSON:
  - `step_iocorrect_outputs_YYYYMMDD/SBxxx/step5a_solution.json`
- Diagnostic quicklook(s) stored in the same output folder.

This solution is then used by **Step5B** to correct FITS WCS for selected/all frames.

In [9]:
import sys
sys.path.insert(0, "/data/jzhang/nenufarIMG_workflow")

from nenufar_ui import run_step5a_iocorrect_solve_ui

run_step5a_iocorrect_solve_ui(
    step4_root="/data/jzhang/nenufar_workflows/step4_outputs_20240310",
    # out_root: /data/jzhang/nenufar_workflows/step_iocorrect_outputs_YYYYMMDD
    default_sb="SB410",
)

## Step5B — Apply Step5A solution (write corrected FITS + before/after quicklooks)

Step5B applies the Step5A WCS shift solution to FITS images and writes **corrected FITS**.
It can process either:
- manually selected FITS (mouse multi-select), or
- **Select ALL image.fits** for batch processing

**How to use**
1. Ensure Step5A has produced `step5a_solution.json` for this SB.
2. Run the Step5B cell to open the widget UI.
3. In the UI:
   - choose SB
   - select FITS or enable **Select ALL image.fits**
   - optionally enable before/after quicklook generation and movie
   - click **Run Step5B (apply)**

**Outputs**
- Corrected FITS:
  - `step_iocorrect_outputs_YYYYMMDD/SBxxx/corr_fits/*.fits`
- Before/after comparison PNGs:
  - `step_iocorrect_outputs_YYYYMMDD/SBxxx/quicklook_step5b/`
- Optional movie (if enabled)
- Log:
  - `step_iocorrect_outputs_YYYYMMDD/SBxxx/step5b_apply.log`

In [ ]:
import sys
sys.path.insert(0, "/data/jzhang/nenufarIMG_workflow")

from nenufar_ui import run_step5b_iocorrect_apply_ui

run_step5b_iocorrect_apply_ui(
    step4_root="/data/jzhang/nenufar_workflows/step4_outputs_20240310",
    # out_root: /data/jzhang/nenufar_workflows/step_iocorrect_outputs_YYYYMMDD
    default_sb="SB410",
)

## Step5C — Centroid tool (raw Step4 FITS or corrected Step5B FITS)

Step5C measures radio-source centroid positions using a **2D Gaussian fit** within a user-defined ROI.
It works on:
- **Step4 raw** `*-image.fits`, or
- **Step5B corrected** FITS under `corr_fits/`

**How to use**
1. Run the Step5C cell to open the widget UI.
2. In the UI:
   - choose **Source**: `Step4 raw` or `Step5B corrected`
   - select FITS file(s) (or enable **Select ALL image.fits**)
   - define ROI bounds (arcsec): xmin/xmax/ymin/ymax
   - set fit controls (`thresh_frac`, `min_points`)
   - optionally enable **Show images inline** (limit by Max images) and/or **Make movie**
   - click **Run Step5C (centroid)**

**Outputs**
- Quicklook PNG(s):
  - `step_iocentroid_outputs_YYYYMMDD/.../quicklook_centroid/`
- Table (CSV):
  - `centroid_results.csv`
- Records (JSONL):
  - `centroid_results.jsonl`
- Log:
  - `step5c_centroid.log`
- Optional movie (if enabled): mp4 (or GIF fallback)

Each record includes (when available): date tag, SB, FITS filename (t-index), observation time, ROI bounds, and centroid coordinates (arcsec / world coordinates).

In [11]:
import sys
sys.path.insert(0, "/data/jzhang/nenufarIMG_workflow")

from nenufar_ui import run_step5c_centroid_ui

run_step5c_centroid_ui(
    step4_root="/data/jzhang/nenufar_workflows/step4_outputs_20240310",
    # step5b_root auto find /data/jzhang/nenufar_workflows/step_iocorrect_outputs_YYYYMMDD 中最新的那个
    # step5b_root="/data/jzhang/nenufar_workflows/step_iocorrect_outputs_20260225",
    default_sb="SB410",
)